# 01 — SimFin Historical Data Pipeline

**Concentric Value Model · Empirical Validation · DBA Research**

This notebook downloads SimFin's bulk fundamentals, builds the point-in-time firm-quarter panel covering **2013-Q1 through 2023-Q4 (44 quarters × 200 firms = 8,800 firm-quarter observations)**, and saves it to Google Drive for use by notebooks 02 and 03.

**Run time:** ~12 minutes on first run (SimFin bulk download); ~30 seconds on subsequent runs (Drive cache).

**Methodology guarantees:**
- Every observation uses ONLY data publicly available at that quarter-end (SimFin `publish_date` filter)
- PE, PB, market_cap, beta are properly derived (not approximated)
- ROE uses trailing-twelve-month net income (requires all 4 quarters available)
- Forward returns use pandas business-day calendar AND verify the horizon endpoint is reachable
- Financial-sector tickers (banks + insurance) are loaded via SimFin's specialized schemas
- Market-index availability is explicitly verified with fallback to equal-weight proxy

## Step 1 — Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/cvm-research'
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(f'{WORK_DIR}/data', exist_ok=True)
os.makedirs(f'{WORK_DIR}/output', exist_ok=True)
print(f'Working directory: {WORK_DIR}')

In [ ]:
!pip install simfin pyarrow --quiet
print('Dependencies installed.')

In [ ]:
REPO_PATH = 'FelixDLanger/cvm-research'  # change here if repo is forked/renamed

!git clone https://github.com/{REPO_PATH}.git /content/cvm-research 2>/dev/null || (cd /content/cvm-research && git pull)

import sys
sys.path.insert(0, '/content/cvm-research')
from cvm_research.scoring import score_panel, DIMENSIONS, LAYERS
from cvm_research.pit_data import (
    quarter_ends, TickerIndex,
    build_pit_panel, attach_forward_returns, market_index_daily_returns,
    cap_tier_from_market_cap,
)
import cvm_research
print(f'cvm_research v{cvm_research.__version__} loaded.')

## Step 2 — Configure SimFin

Free API key from [simfin.com](https://simfin.com) → dashboard. Add to Colab Secrets (key icon → name: `SIMFIN_API_KEY`).

In [ ]:
from google.colab import userdata
try:
    SIMFIN_API_KEY = userdata.get('SIMFIN_API_KEY')
except Exception as e:
    raise RuntimeError(
        'Add your SimFin API key to Colab Secrets (left sidebar → key icon → '
        'name: SIMFIN_API_KEY, value: <your key>)'
    ) from e

import simfin as sf
sf.set_api_key(SIMFIN_API_KEY)
sf.set_data_dir(f'{WORK_DIR}/data')
print('SimFin configured.')

## Step 3 — Download SimFin bulk CSVs (with banks + insurance schemas)

**Fix applied (audit item #2):** SimFin separates banks and insurance into specialized statement schemas because their reporting structure differs from non-financial firms. Without explicitly loading these, all 41 FIN-sector tickers in our universe (JPM, BAC, V, MA, WFC, GS, BLK, MS, AXP, SPGI, etc.) would silently fall back to sector baseline scoring — a 20% silent degradation of the panel.

This cell loads the standard schema PLUS banks PLUS insurance, then concatenates.

In [ ]:
import pandas as pd

markets = ['us', 'de', 'uk', 'fr', 'jp', 'cn']
per_market_data = {}

def safe_load(loader, **kwargs):
    """Call a simfin loader, returning empty DataFrame on failure."""
    try:
        return loader(**kwargs)
    except Exception as e:
        return pd.DataFrame()

for mkt in markets:
    try:
        # Companies (one file per market)
        companies_mkt = safe_load(sf.load_companies, market=mkt)
        if len(companies_mkt) == 0:
            print(f'  {mkt.upper()}: SKIPPED — no companies file')
            continue

        # Income / balance / cashflow: standard schema (non-financials)
        income_std    = safe_load(sf.load_income,   variant='quarterly', market=mkt)
        balance_std   = safe_load(sf.load_balance,  variant='quarterly', market=mkt)
        cashflow_std  = safe_load(sf.load_cashflow, variant='quarterly', market=mkt)

        # Banks specialized schema
        income_banks   = safe_load(sf.load_income_banks,   variant='quarterly', market=mkt)
        balance_banks  = safe_load(sf.load_balance_banks,  variant='quarterly', market=mkt)
        cashflow_banks = safe_load(sf.load_cashflow_banks, variant='quarterly', market=mkt)

        # Insurance specialized schema
        income_ins   = safe_load(sf.load_income_insurance,   variant='quarterly', market=mkt)
        balance_ins  = safe_load(sf.load_balance_insurance,  variant='quarterly', market=mkt)
        cashflow_ins = safe_load(sf.load_cashflow_insurance, variant='quarterly', market=mkt)

        # Concatenate the three schemas — column unions handle the differences
        income_mkt   = pd.concat([income_std, income_banks, income_ins], ignore_index=True)
        balance_mkt  = pd.concat([balance_std, balance_banks, balance_ins], ignore_index=True)
        cashflow_mkt = pd.concat([cashflow_std, cashflow_banks, cashflow_ins], ignore_index=True)

        prices_mkt = safe_load(sf.load_shareprices, variant='daily', market=mkt)

        per_market_data[mkt] = {
            'companies': companies_mkt,
            'income':    income_mkt,
            'balance':   balance_mkt,
            'cashflow':  cashflow_mkt,
            'prices':    prices_mkt,
        }
        print(f'  {mkt.upper()}: {len(companies_mkt):>5} companies · '
              f'{len(income_std):>5} std + {len(income_banks):>4} banks + {len(income_ins):>4} ins income · '
              f'{len(prices_mkt):>7} prices')
    except Exception as e:
        print(f'  {mkt.upper()}: SKIPPED — {e}')

print('Bulk download complete.')

## Step 4 — Normalise and concatenate

In [ ]:
def normalize_simfin_df(df):
    df = df.reset_index() if hasattr(df, 'reset_index') else df
    if 'Ticker' in df.columns: df = df.rename(columns={'Ticker': 'ticker'})
    if 'Publish Date' in df.columns: df['publish_date'] = pd.to_datetime(df['Publish Date'], errors='coerce')
    if 'Report Date' in df.columns: df['report_date'] = pd.to_datetime(df['Report Date'], errors='coerce')
    if 'Date' in df.columns: df['date'] = pd.to_datetime(df['Date'], errors='coerce')
    if 'Adj. Close' in df.columns: df = df.rename(columns={'Adj. Close': 'adj_close'})
    if 'Close' in df.columns and 'close' not in df.columns: df = df.rename(columns={'Close': 'close'})
    if 'Volume' in df.columns: df = df.rename(columns={'Volume': 'volume'})
    if 'Dividend' in df.columns: df = df.rename(columns={'Dividend': 'dividend'})
    if 'Shares Outstanding' in df.columns: df = df.rename(columns={'Shares Outstanding': 'shares_out'})
    return df

all_income, all_balance, all_cashflow, all_prices, all_companies = [], [], [], [], []
for mkt, d in per_market_data.items():
    all_companies.append(normalize_simfin_df(d['companies']))
    all_income.append(normalize_simfin_df(d['income']))
    all_balance.append(normalize_simfin_df(d['balance']))
    all_cashflow.append(normalize_simfin_df(d['cashflow']))
    all_prices.append(normalize_simfin_df(d['prices']))

all_companies = pd.concat(all_companies, ignore_index=True)
all_income    = pd.concat(all_income, ignore_index=True)
all_balance   = pd.concat(all_balance, ignore_index=True)
all_cashflow  = pd.concat(all_cashflow, ignore_index=True)
all_prices    = pd.concat(all_prices, ignore_index=True)

print(f'Combined: {len(all_companies):,} companies · {len(all_income):,} income · {len(all_prices):,} prices')

## Step 5 — Build per-ticker indexes (O(log N) lookups)

In [ ]:
print('Building per-ticker indexes...')
income_idx   = TickerIndex(all_income,   date_col='publish_date')
balance_idx  = TickerIndex(all_balance,  date_col='publish_date')
cashflow_idx = TickerIndex(all_cashflow, date_col='publish_date')
price_idx    = TickerIndex(all_prices,   date_col='date')
print(f'  income:   {len(income_idx._per_ticker):,} tickers indexed')
print(f'  balance:  {len(balance_idx._per_ticker):,} tickers indexed')
print(f'  cashflow: {len(cashflow_idx._per_ticker):,} tickers indexed')
print(f'  price:    {len(price_idx._per_ticker):,} tickers indexed')

## Step 6 — Load reference universe + sector normalisation

In [ ]:
import urllib.request

REPO_FOR_TICKERS = 'FelixDLanger/concentric-value-model'
tickers_url = f'https://raw.githubusercontent.com/{REPO_FOR_TICKERS}/main/tickers.csv'
tickers_path = f'{WORK_DIR}/tickers.csv'
urllib.request.urlretrieve(tickers_url, tickers_path)

# Normalise Phase 1 sector codes to Phase 2 scoring engine codes
SECTOR_NORMALISE = {'HEALTH': 'HC', 'IND': 'INDUS', 'CONS_S': 'CONS_D', 'REAL': 'RE'}

ref_universe = []
with open(tickers_path) as f:
    for line in f:
        if line.startswith('#') or line.startswith('ticker,'):
            continue
        parts = line.strip().split(',')
        if len(parts) >= 5:
            sector = SECTOR_NORMALISE.get(parts[4], parts[4])
            ref_universe.append({
                'ticker': parts[0], 'name': parts[1],
                'country': parts[2], 'region': parts[3], 'sector': sector,
            })

ref_df = pd.DataFrame(ref_universe)
print(f'Loaded {len(ref_df)} tickers across {ref_df["region"].nunique()} regions')
print(ref_df.groupby(['region','sector']).size().unstack(fill_value=0))

## Step 7 — Market index returns (for beta) — with availability check

**Fix applied (audit item #5):** explicitly verify SPY is available before relying on it. If SimFin's bulk download doesn't include it, fall back to the equal-weighted return of our 200-firm universe as a market proxy. This is a defensible alternative that maintains the "all five controls" guarantee.

In [ ]:
# First try SPY (canonical market proxy)
mkt_returns = market_index_daily_returns(price_idx, index_ticker='SPY')

if len(mkt_returns) > 100:
    market_proxy_source = 'SPY'
    print(f'✓ Using SPY as market proxy: {len(mkt_returns):,} daily returns from '
          f'{mkt_returns.index.min().date()} to {mkt_returns.index.max().date()}')
else:
    # Fallback: equal-weighted return of the 200-firm universe
    print(f'⚠ SPY unavailable in price data ({len(mkt_returns)} returns). '
          'Falling back to equal-weighted universe return as market proxy.')
    market_proxy_source = 'EW_UNIVERSE'

    universe_returns_by_date = {}
    for t in ref_df['ticker'].tolist():
        sub = price_idx._per_ticker.get(t)
        if sub is None or len(sub) < 2:
            continue
        cc = 'adj_close' if 'adj_close' in sub.columns else 'close'
        s = sub.set_index('date')[cc].dropna().sort_index().pct_change().dropna()
        for d, r in s.items():
            universe_returns_by_date.setdefault(d, []).append(r)

    mkt_returns = pd.Series({
        d: sum(rs) / len(rs)
        for d, rs in universe_returns_by_date.items()
        if len(rs) >= 50  # need at least 50 firms reporting on that day for a meaningful average
    }).sort_index()
    print(f'  EW-universe proxy: {len(mkt_returns):,} daily returns')

# This source label gets included in regression output for transparency
print(f'\nMarket proxy source for beta computation: {market_proxy_source}')

## Step 8 — Build the point-in-time panel

In [ ]:
from cvm_research.pit_data import quarter_ends

QUARTERS = quarter_ends(2013, 2023)
print(f'{len(QUARTERS)} quarter-ends from {QUARTERS[0].date()} to {QUARTERS[-1].date()}')

ticker_to_meta = {
    row['ticker']: {
        'sector': row['sector'], 'country': row['country'], 'name': row['name'],
        'cap': None,
    }
    for _, row in ref_df.iterrows()
}

print('\nBuilding panel (3-5 minutes)...')
panel = build_pit_panel(
    tickers=ref_df['ticker'].tolist(),
    quarter_dates=QUARTERS,
    income_idx=income_idx, balance_idx=balance_idx,
    cashflow_idx=cashflow_idx, price_idx=price_idx,
    ticker_to_meta=ticker_to_meta,
    market_index_returns=mkt_returns,
)
print(f'\nPanel built: {len(panel):,} firm-quarter observations')
print()
print('Coverage of key fields:')
for col in ['pe', 'pb', 'market_cap', 'beta', 'roe', 'debt_equity', 'price_in_range']:
    n_present = panel[col].notna().sum() if col in panel.columns else 0
    print(f'  {col:20s}: {n_present:5d} / {len(panel):5d} ({100*n_present/len(panel):4.1f}%)')

## Step 9 — Attach forward 1-year returns (with horizon validation)

**Fix applied (audit item #3):** `forward_total_return` now verifies that the end date is actually reachable in the price data. If `as_of + 252 trading days` falls beyond the available price series, the function returns None rather than silently producing a partial-horizon return. This eliminates the late-sample downward bias.

In [ ]:
print('Computing forward 252-trading-day returns (with horizon validation)...')
panel_with_returns = attach_forward_returns(panel, price_idx, trading_days=252)

n_fwd = panel_with_returns['forward_return_pct'].notna().sum()
print(f'Forward returns: {n_fwd:,} / {len(panel_with_returns):,} ({100*n_fwd/len(panel_with_returns):.1f}%)')

# Diagnostic: distribution across as_of quarters (late-sample drop is expected and correct)
print()
print('Forward-return coverage by year (late quarters legitimately drop when horizon exceeds data):')
panel_with_returns['year'] = pd.to_datetime(panel_with_returns['as_of']).dt.year
coverage_by_year = panel_with_returns.groupby('year').agg(
    n_obs=('ticker','count'),
    n_fwd_returns=('forward_return_pct', lambda s: s.notna().sum()),
).assign(coverage_pct=lambda d: (d['n_fwd_returns']/d['n_obs']*100).round(1))
print(coverage_by_year)

## Step 10 — Country macro time series

In [ ]:
import requests

WB_INDICATORS = {'gdp_growth': 'NY.GDP.MKTP.KD.ZG', 'inflation': 'FP.CPI.TOTL.ZG'}
WB_COUNTRY_MAP = {
    'US':'USA','DE':'DEU','NL':'NLD','FR':'FRA','DK':'DNK','CH':'CHE','IT':'ITA',
    'ES':'ESP','BE':'BEL','GB':'GBR','JP':'JPN','TW':'TWN','KR':'KOR','CN':'CHN',
    'HK':'HKG','IN':'IND','SG':'SGP',
}

macro_records = []
for cc2, cc3 in WB_COUNTRY_MAP.items():
    for label, indicator in WB_INDICATORS.items():
        url = f'https://api.worldbank.org/v2/country/{cc3}/indicator/{indicator}'
        try:
            r = requests.get(url, params={'format':'json', 'date':'2013:2023', 'per_page':'30'}, timeout=15)
            data = r.json()
            if isinstance(data, list) and len(data) > 1:
                for rec in data[1] or []:
                    if rec.get('value') is not None:
                        macro_records.append({
                            'country':cc2, 'year':int(rec['date']),
                            'indicator':label, 'value':float(rec['value']),
                        })
        except Exception as e:
            print(f'  WB fetch failed for {cc3}/{label}: {e}')

macro_long = pd.DataFrame(macro_records)
macro_wide = macro_long.pivot_table(index=['country','year'], columns='indicator',
                                    values='value').reset_index()
print(f'Macro data: {len(macro_wide)} country-years')

In [ ]:
# Merge macro by (country, year-of-as_of)
panel_with_returns['year'] = pd.to_datetime(panel_with_returns['as_of']).dt.year
panel_full = panel_with_returns.merge(macro_wide, on=['country','year'], how='left')
panel_full = panel_full.rename(columns={'gdp_growth':'macro_gdp', 'inflation':'macro_inflation'})
print(f'Panel with macro: {len(panel_full):,} rows')

## Step 11 — Save to Drive

In [ ]:
panel_full.to_parquet(f'{WORK_DIR}/output/panel_pit_2013_2023.parquet')
macro_wide.to_parquet(f'{WORK_DIR}/output/macro_history.parquet')
panel_full.to_csv(f'{WORK_DIR}/output/panel_pit_2013_2023.csv', index=False)

# Also save a metadata record for transparency
import json
metadata = {
    'build': 'phase2-v2.1-fixes',
    'sample_period': '2013-Q1 to 2023-Q4',
    'n_quarters': len(QUARTERS),
    'n_tickers_universe': len(ref_df),
    'n_observations': len(panel_full),
    'n_forward_returns_available': int(panel_full['forward_return_pct'].notna().sum()),
    'market_proxy_source': market_proxy_source,
    'fixes_applied': [
        'Sector keys normalised to Phase 2 scoring engine (HEALTH→HC, IND→INDUS, CONS_S→CONS_D, REAL→RE)',
        'Banks + insurance schemas loaded for FIN sector (41 tickers, ~20% of universe)',
        'TTM net income requires all 4 quarters (not 3)',
        'Forward returns validated against data-horizon availability',
        'Market proxy availability check with EW-universe fallback',
    ],
}
with open(f'{WORK_DIR}/output/panel_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print('Saved:')
print(f'  panel_pit_2013_2023.parquet ({len(panel_full):,} rows)')
print(f'  macro_history.parquet ({len(macro_wide):,} rows)')
print(f'  panel_metadata.json (provenance record)')

## Summary

You now have a methodologically rigorous point-in-time panel:

- **44 quarters × 200 firms = up to 8,800 observations**
- **Effective N varies by field** — see the coverage table above; financial-sector tickers now have proper coverage thanks to the banks+insurance schema load
- **Forward returns** are honest 1-year returns (no partial-horizon contamination at the late edge of the sample)
- **Beta** uses a defensible market proxy (SPY if available, equal-weighted universe otherwise — provenance recorded in `panel_metadata.json`)
- **TTM ROE** requires all 4 quarters of net income (no 33%-biased pseudo-TTM)

**Next:** open `02_pca_analysis.ipynb` for Q-B, then `03_backtest_regression.ipynb` for Q-A.